# HeyTea CNN Training — Kaggle Notebook

**Before running:**
1. Upload `tea_dataset/` images as Kaggle Dataset named **`heytea-images`**
2. Add that dataset to this notebook (+ Data → Add dataset)
3. Add your W&B API key: Settings → Secrets → `WANDB_API_KEY`
4. Enable GPU: Settings → Accelerator → GPU T4 x2 (or P100)
5. Enable Internet: Settings → Internet → On

**Run order:**
- Set `RUN_SWEEP = True` first (find best hyperparams, ~4–6 hr)
- Then update `BEST_HPARAMS` in the config cell below
- Re-run with `RUN_SWEEP = False` for final 5-seed evaluation

In [ ]:
# ── Install extra packages ────────────────────────────────────────────────────
# torch/torchvision already available on Kaggle
!pip install -q wandb scikit-learn scipy

In [ ]:
# ── Clone repo for training code ──────────────────────────────────────────────
import subprocess
result = subprocess.run(
    ['git', 'clone', 'https://github.com/tanpim16/dl_dads7202_heytea', '/kaggle/working/repo'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

In [ ]:
# ── Override paths for Kaggle filesystem ─────────────────────────────────────
import sys
from pathlib import Path

sys.path.insert(0, '/kaggle/working/repo/training')

import config
config.DATA_DIR       = Path('/kaggle/input/heytea-images/tea_dataset')
config.METADATA       = config.DATA_DIR / 'metadata_clean.csv'
config.CHECKPOINT_DIR = Path('/kaggle/working/checkpoints')
config.RESULTS_DIR    = Path('/kaggle/working/results')

config.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_DIR:', config.DATA_DIR)
print('Exists? ', config.DATA_DIR.exists())

In [ ]:
# ── (Optional) Override BEST_HPARAMS after sweep ─────────────────────────────
# Fill these in from the W&B sweep results before final run
config.BEST_HPARAMS = {
    'vgg16': {
        'stage1_lr': 0.001,  'stage1_epochs': 10,
        'stage2_lr': 0.0001, 'stage2_epochs': 15,
        'label_smoothing': 0.05,
    },
    'resnet50': {
        'stage1_lr': 0.001,  'stage1_epochs': 10,
        'stage2_lr': 0.0001, 'stage2_epochs': 15,
        'label_smoothing': 0.0,
    },
    'efficientnet_b3': {
        'stage1_lr': 0.001,  'stage1_epochs': 10,
        'stage2_lr': 0.0001, 'stage2_epochs': 15,
        'label_smoothing': 0.0,
    },
    'mobilenet_v3_large': {
        'stage1_lr': 0.001,  'stage1_epochs': 5,
        'stage2_lr': 0.00005,'stage2_epochs': 20,
        'label_smoothing': 0.0,
    },
}

In [ ]:
# ── W&B login ─────────────────────────────────────────────────────────────────
import os
import wandb
from kaggle_secrets import UserSecretsClient

secret = UserSecretsClient()
os.environ['WANDB_API_KEY'] = secret.get_secret('WANDB_API_KEY')
wandb.login()
print('W&B logged in')

In [ ]:
# ── Verify split + data ───────────────────────────────────────────────────────
from dataset import make_splits, make_loaders, get_class_weights

train_df, val_df, test_df = make_splits()
print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')
print('\nTrain class counts:')
print(train_df.groupby('class').size().to_string())
print('\nTest class counts:')
print(test_df.groupby('class').size().to_string())

In [ ]:
# ── Option A: Run W&B sweep (set RUN_SWEEP=True, ~4-6 hr on T4) ───────────────
RUN_SWEEP = False   # <- set True first time

if RUN_SWEEP:
    from run_sweep import SWEEP_CONFIG, sweep_train
    sweep_id = wandb.sweep(SWEEP_CONFIG, project='heytea-cnn')
    wandb.agent(sweep_id, function=sweep_train, count=40)  # 40 runs on Kaggle
    print('Sweep done. Check W&B for best hyperparams, then update BEST_HPARAMS above and re-run with RUN_SWEEP=False')

In [ ]:
# ── Option B: Final 5-seed run (set RUN_SWEEP=False) ─────────────────────────
if not RUN_SWEEP:
    from run_final import run_all
    run_all()
    print('\nDone! Results saved to /kaggle/working/results/')

In [ ]:
# ── Show saved result files ───────────────────────────────────────────────────
import os
results_dir = Path('/kaggle/working/results')
for f in sorted(results_dir.iterdir()):
    print(f.name, f'({f.stat().st_size // 1024} KB)')